In [133]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/Users/emmanuel/Documents/belugas/beluga-call-pipeline
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [134]:
import pandas as pd
labels_v1 = pd.read_csv("../../data/Verified_Dataset/labels/labels_eval_v1.csv")
irene_df = pd.read_csv("../../data/Verified_Dataset/labels/labels_irene_20min.csv")
old_abs_df = pd.read_csv("../../data/Verified_Dataset/labels/labels_old_abs.csv")

In [135]:
labels_v1["labeling_effort"] = "evaluation_v1"
irene_df["labeling_effort"] = "irene_20min"

In [136]:
old_abs_df["labeling_effort"] = "old_abs_from_3s_snippets"
old_abs_df.loc[old_abs_df["Origin"] == "1s_absences", "labeling_effort"] = "old_abs_CAC_last_samples"
old_abs_df.drop(columns=["Origin"], inplace=True)

In [137]:
pd.set_option('display.max_columns', None)
# Compare columns of the three dataframes and print the differences

v1_cols = set(labels_v1.columns)
irene_cols = set(irene_df.columns)
old_abs_cols = set(old_abs_df.columns)

# Compare labels_v1 vs irene_df
v1_not_in_irene = v1_cols - irene_cols
irene_not_in_v1 = irene_cols - v1_cols

if v1_not_in_irene:
    print("Columns in labels_v1 but not in irene_df:", v1_not_in_irene)
else:
    print("No columns are unique to labels_v1 with respect to irene_df.")

if irene_not_in_v1:
    print("Columns in irene_df but not in labels_v1:", irene_not_in_v1)
else:
    print("No columns are unique to irene_df with respect to labels_v1.")

# Compare labels_v1 vs old_abs_df
v1_not_in_oldabs = v1_cols - old_abs_cols
oldabs_not_in_v1 = old_abs_cols - v1_cols

if v1_not_in_oldabs:
    print("Columns in labels_v1 but not in old_abs_df:", v1_not_in_oldabs)
else:
    print("No columns are unique to labels_v1 with respect to old_abs_df.")

if oldabs_not_in_v1:
    print("Columns in old_abs_df but not in labels_v1:", oldabs_not_in_v1)
else:
    print("No columns are unique to old_abs_df with respect to labels_v1.")

# Compare irene_df vs old_abs_df
irene_not_in_oldabs = irene_cols - old_abs_cols
oldabs_not_in_irene = old_abs_cols - irene_cols

if irene_not_in_oldabs:
    print("Columns in irene_df but not in old_abs_df:", irene_not_in_oldabs)
else:
    print("No columns are unique to irene_df with respect to old_abs_df.")

if oldabs_not_in_irene:
    print("Columns in old_abs_df but not in irene_df:", oldabs_not_in_irene)
else:
    print("No columns are unique to old_abs_df with respect to irene_df.")


Columns in labels_v1 but not in irene_df: {'Begin Time (s)', 'labeled_snippet_filename', 'labeled_snippet_dir', 'original_filename', 'snippet_end_s', 'snippet_start_s', 'End Time (s)'}
Columns in irene_df but not in labels_v1: {'start_s', 'end_s', 'DETAIL', 'SnippetFilename'}
Columns in labels_v1 but not in old_abs_df: {'Begin Time (s)', 'labeled_snippet_filename', 'labeled_snippet_dir', 'Notes', 'snippet_end_s', 'GROUNDTRUTH', 'snippet_start_s', 'End Time (s)'}
Columns in old_abs_df but not in labels_v1: {'start_s', 'end_s', 'SnippetFilename'}
Columns in irene_df but not in old_abs_df: {'Notes', 'DETAIL', 'GROUNDTRUTH'}
Columns in old_abs_df but not in irene_df: {'original_filename'}


In [138]:
# Merge the two dataframes, keeping all columns, filling missing values with empty where necessary
merged_df = pd.concat([labels_v1, irene_df, old_abs_df], ignore_index=True, sort=False)
# Fill all NaN values with empty string for reporting/compatibility
merged_df = merged_df.fillna('')
# Show info and preview to check result
print("Merged DataFrame shape:", merged_df.shape)
display(merged_df.head())

Merged DataFrame shape: (11669, 26)


,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,clip_filename,ECHO,BBPC,HFPC,Whistle,Boat,labeling_effort,DETAIL,SnippetFilename,start_s,end_s
0,0.0,1.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:12.000000000,2017-07-24 14:09:13.000000000,BSM_20170724_14091200.wav,1,0,0,0,0.0,evaluation_v1,,,,
1,1.0,2.0,a,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:13.000000000,2017-07-24 14:09:14.000000000,BSM_20170724_14091300.wav,0,0,0,0,0.0,evaluation_v1,,,,
2,2.0,3.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:14.000000000,2017-07-24 14:09:15.000000000,BSM_20170724_14091400.wav,1,0,0,0,0.0,evaluation_v1,,,,
3,3.0,4.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:15.000000000,2017-07-24 14:09:16.000000000,BSM_20170724_14091500.wav,1,0,0,0,0.0,evaluation_v1,,,,
4,4.0,5.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:16.000000000,2017-07-24 14:09:17.000000000,BSM_20170724_14091600.wav,1,0,0,0,0.0,evaluation_v1,,,,


In [139]:
merged_df["Call"] = merged_df[["ECHO", "BBPC", "HFPC", "Whistle"]].any(axis=1).astype(int)


In [140]:
# Define the desired order for the first 7 columns
first_columns = ['clip_filename', 'ECHO', 'BBPC', 'HFPC', 'Whistle', 'Call', 'Boat', 'labeling_effort', 'DETAIL']
# Get the remaining columns not in the first_columns list
remaining_columns = [col for col in merged_df.columns if col not in first_columns]
# Reorder the dataframe
merged_df = merged_df[first_columns + remaining_columns]
merged_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s
0,BSM_20170724_14091200.wav,1,0,0,0,1,0.0,evaluation_v1,,0.0,1.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:12.000000000,2017-07-24 14:09:13.000000000,,,
1,BSM_20170724_14091300.wav,0,0,0,0,0,0.0,evaluation_v1,,1.0,2.0,a,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:13.000000000,2017-07-24 14:09:14.000000000,,,
2,BSM_20170724_14091400.wav,1,0,0,0,1,0.0,evaluation_v1,,2.0,3.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:14.000000000,2017-07-24 14:09:15.000000000,,,
3,BSM_20170724_14091500.wav,1,0,0,0,1,0.0,evaluation_v1,,3.0,4.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:15.000000000,2017-07-24 14:09:16.000000000,,,
4,BSM_20170724_14091600.wav,1,0,0,0,1,0.0,evaluation_v1,,4.0,5.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:16.000000000,2017-07-24 14:09:17.000000000,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11664,CAC_20210714_09242300.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:23.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:23.000000,2021-07-14 09:24:24.000000,CAC_20210714_09242300.wav,-1.0,0.0
11665,CAC_20210714_09242800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:28.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:28.000000,2021-07-14 09:24:29.000000,CAC_20210714_09242800.wav,-1.0,0.0
11666,CAC_20210714_09251600.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:25:16.000000,,,CAC,,201359382,-172.7,2021-07-14 09:25:16.000000,2021-07-14 09:25:17.000000,CAC_20210714_09251600.wav,-1.0,0.0
11667,CAC_20210714_09271800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:27:18.000000,,,CAC,,201359382,-172.7,2021-07-14 09:27:18.000000,2021-07-14 09:27:19.000000,CAC_20210714_09271800.wav,-1.0,0.0


In [141]:
print(merged_df["ECHO"].value_counts())
print(merged_df["BBPC"].value_counts())
print(merged_df["HFPC"].value_counts())
print(merged_df["Whistle"].value_counts())
print(merged_df["Call"].value_counts())
print(merged_df["Boat"].value_counts())


ECHO
0    8106
1    3563
Name: count, dtype: int64
BBPC
0    10926
1      743
Name: count, dtype: int64
HFPC
0    11034
1      635
Name: count, dtype: int64
Whistle
0    9685
1    1984
Name: count, dtype: int64
Call
0    7406
1    4263
Name: count, dtype: int64
Boat
       6869
0.0    2939
1.0    1861
Name: count, dtype: int64


In [142]:
merged_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s
0,BSM_20170724_14091200.wav,1,0,0,0,1,0.0,evaluation_v1,,0.0,1.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:12.000000000,2017-07-24 14:09:13.000000000,,,
1,BSM_20170724_14091300.wav,0,0,0,0,0,0.0,evaluation_v1,,1.0,2.0,a,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:13.000000000,2017-07-24 14:09:14.000000000,,,
2,BSM_20170724_14091400.wav,1,0,0,0,1,0.0,evaluation_v1,,2.0,3.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:14.000000000,2017-07-24 14:09:15.000000000,,,
3,BSM_20170724_14091500.wav,1,0,0,0,1,0.0,evaluation_v1,,3.0,4.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:15.000000000,2017-07-24 14:09:16.000000000,,,
4,BSM_20170724_14091600.wav,1,0,0,0,1,0.0,evaluation_v1,,4.0,5.0,e,,201359382.170724133002.wav,201359382.170724140912.snippet.wav,2017-07-24 14:09:12,2350.0,2950.0,BSM,../../data/evaluation_snippets/v1/BSM_2017,201359382,-172.7,2017-07-24 14:09:16.000000000,2017-07-24 14:09:17.000000000,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11664,CAC_20210714_09242300.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:23.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:23.000000,2021-07-14 09:24:24.000000,CAC_20210714_09242300.wav,-1.0,0.0
11665,CAC_20210714_09242800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:24:28.000000,,,CAC,,201359382,-172.7,2021-07-14 09:24:28.000000,2021-07-14 09:24:29.000000,CAC_20210714_09242800.wav,-1.0,0.0
11666,CAC_20210714_09251600.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:25:16.000000,,,CAC,,201359382,-172.7,2021-07-14 09:25:16.000000,2021-07-14 09:25:17.000000,CAC_20210714_09251600.wav,-1.0,0.0
11667,CAC_20210714_09271800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 09:27:18.000000,,,CAC,,201359382,-172.7,2021-07-14 09:27:18.000000,2021-07-14 09:27:19.000000,CAC_20210714_09271800.wav,-1.0,0.0


# Verifying all wavs are generated and 1 second long

In [143]:
# Only consider duplicates for rows where labeling_effort is "irene_20min" or "evaluation_v1"
# effort_mask = merged_df["labeling_effort"].isin(["irene_20min", "evaluation_v1"])
effort_mask = merged_df["labeling_effort"].isin(["irene_20min", "evaluation_v1", "old_abs_CAC_last_samples", "old_abs_from_3s_snippets"])
effort_df = merged_df[effort_mask]

dupes_restricted = effort_df[effort_df['clip_filename'].duplicated(keep=False)].copy()

# Columns to check for label consistency
cols_to_check_restricted = ['ECHO', 'BBPC', 'HFPC', 'Whistle']

def has_mismatched_labels_restricted(group):
    # True if at least two rows in the group have different label values across the columns of interest
    return group[cols_to_check_restricted].nunique(dropna=False).max() > 1

# Compute mismatch flag for each group using apply(), then map back to the DataFrame
mismatch_by_clip = dupes_restricted.groupby('clip_filename').apply(has_mismatched_labels_restricted)
dupes_restricted["label_mismatch"] = dupes_restricted['clip_filename'].map(mismatch_by_clip)

print(dupes_restricted["label_mismatch"].value_counts())
sorted_dupes = dupes_restricted.sort_values(by='clip_filename')
display(sorted_dupes[sorted_dupes["clip_filename"] == "CAC_20210714_08384800.wav"].head(20))

diff_groups_restricted = dupes_restricted[dupes_restricted["label_mismatch"] == True]

if not diff_groups_restricted.empty:
    print("Duplicated clip_filenames (restricted to irene_20min or evaluation_v1) with differing ECHO, BBPC, HFPC, or Whistle values:")
    diff_groups_restricted = diff_groups_restricted.sort_values(by='clip_filename')
    display(diff_groups_restricted.head(10))
else:
    print("All duplicated clip_filenames (irene_20min or evaluation_v1) have identical ECHO, BBPC, HFPC, and Whistle values.")

label_mismatch
False    86
True     34
Name: count, dtype: int64


/var/folders/24/7c98sw1j4_13xl92r115m6640000gn/T/ipykernel_7350/571713558.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mismatch_by_clip = dupes_restricted.groupby('clip_filename').apply(has_mismatched_labels_restricted)


,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s,label_mismatch
4109,CAC_20210714_08384800.wav,0,0,1,1,1,1.0,irene_20min,hm,,,hw,ship,,,2021-07-14 08:38:47.007000,,,CAC,,201359382,-172.7,2021-07-14 08:38:48.007000,2021-07-14 08:38:49.007000,CAC_20210714_08384700.wav,1.0,2.0,True
3476,CAC_20210714_08384800.wav,1,1,1,1,1,1.0,evaluation_v1,,475.0,476.0,ewbh,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:38:48.000000000,2021-07-14 08:38:49.000000000,,,,True


Duplicated clip_filenames (restricted to irene_20min or evaluation_v1) with differing ECHO, BBPC, HFPC, or Whistle values:


,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s,label_mismatch
3705,BSM_20170724_13562209.wav,1,0,0,0,1,1.0,irene_20min,,,,e,ship,,,2017-07-24 13:56:20.090000,,,BSM,,201359382,-172.7,2017-07-24 13:56:22.090000,2017-07-24 13:56:23.090000,BSM_20170724_13562009.wav,2.0,3.0,True
4806,BSM_20170724_13562209.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 13:56:20.090,,,BSM,,201359382,-172.7,2017-07-24 13:56:22.090,2017-07-24 13:56:23.090,BSM_20170724_13562009.wav,1.0,2.0,True
3785,BSM_20170801_19043620.wav,1,0,0,0,1,0.0,irene_20min,,,,e,,,,2017-08-01 19:04:35.200000,,,BSM,,201359382,-172.7,2017-08-01 19:04:36.200000,2017-08-01 19:04:37.200000,BSM_20170801_19043520.wav,1.0,2.0,True
4816,BSM_20170801_19043620.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-08-01 19:04:35.200,,,BSM,,201359382,-172.7,2017-08-01 19:04:36.200,2017-08-01 19:04:37.200,BSM_20170801_19043520.wav,0.0,1.0,True
11500,CAC_20210714_08305700.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 08:30:57.000000,,,CAC,,201359382,-172.7,2021-07-14 08:30:57.000000,2021-07-14 08:30:58.000000,CAC_20210714_08305700.wav,-1.0,0.0,True
3005,CAC_20210714_08305700.wav,1,0,0,0,1,1.0,evaluation_v1,,4.0,5.0,e,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:30:57.000000000,2021-07-14 08:30:58.000000000,,,,True
11507,CAC_20210714_08314800.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 08:31:48.000000,,,CAC,,201359382,-172.7,2021-07-14 08:31:48.000000,2021-07-14 08:31:49.000000,CAC_20210714_08314800.wav,-1.0,0.0,True
3056,CAC_20210714_08314800.wav,1,0,0,0,1,1.0,evaluation_v1,,55.0,56.0,e,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:31:48.000000000,2021-07-14 08:31:49.000000000,,,,True
3069,CAC_20210714_08320100.wav,1,0,0,0,1,1.0,evaluation_v1,,68.0,69.0,e,,201359382.210714075958.wav,201359382.210714083053.snippet.wav,2021-07-14 08:30:53,1855.0,2455.0,CAC,../../data/evaluation_snippets/v1/CAC_2021,201359382,-172.7,2021-07-14 08:32:01.000000000,2021-07-14 08:32:02.000000000,,,,True
11508,CAC_20210714_08320100.wav,0,0,0,0,0,,old_abs_CAC_last_samples,,,,,,201359382.210714075958.wav,,2021-07-14 08:32:01.000000,,,CAC,,201359382,-172.7,2021-07-14 08:32:01.000000,2021-07-14 08:32:02.000000,CAC_20210714_08320100.wav,-1.0,0.0,True


In [144]:
# Drop duplicates keeping the preferred labeling_effort (priority: evaluation_v1 > irene_20min > old_abs_from_3s_snippets > old_abs_CAC_last_samples)
effort_priority = {
    "evaluation_v1": 0,
    "irene_20min": 1,
    "old_abs_from_3s_snippets": 2,
    "old_abs_CAC_last_samples": 3
}

# Anything not in the map gets lowest priority
merged_df["effort_priority"] = merged_df["labeling_effort"].map(effort_priority).fillna(99).astype(int)

# Sort by clip_filename and effort_priority, so preferred labeling_effort comes first
merged_df = merged_df.sort_values(["clip_filename", "effort_priority"])

# Drop duplicates, keeping the preferred one
merged_df = merged_df.drop_duplicates(subset="clip_filename", keep="first")

# Remove the helper column
merged_df = merged_df.drop(columns=["effort_priority"])

In [145]:

merged_df[merged_df['clip_filename'] == "BSM_20210818_17440200.wav"]

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s
8723,BSM_20210818_17440200.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2021-08-18 17:44:00.000,,,BSM,,201359382,-172.7,2021-08-18 17:44:02.000,2021-08-18 17:44:03.000,BSM_20210818_174400.wav,1.0,2.0


In [146]:
import os
import wave

# The directory where the wavs are located
wav_dir = "../../data/Verified_Dataset/clip_wavs"

# Check for duplicate clip_filenames in the merged_df
dup_clip_filenames = merged_df['clip_filename'][merged_df['clip_filename'] != ""].duplicated(keep=False)
if dup_clip_filenames.any():
    print("Duplicate clip_filenames found:")
    print(merged_df.loc[dup_clip_filenames, 'clip_filename'].value_counts())
else:
    print("No duplicate clip_filenames in merged_df.")

missing_files = []
not_1s_files = []

# Make sure to only use each filename once for efficiency
unique_filenames = merged_df['clip_filename'].unique()

for fname in unique_filenames:
    if not fname:
        continue  # Skip empty filenames

    wav_path = os.path.join(wav_dir, fname)
    if not os.path.isfile(wav_path):
        missing_files.append(fname)
        continue

    try:
        with wave.open(wav_path, 'r') as wf:
            n_frames = wf.getnframes()
            framerate = wf.getframerate()
            duration = n_frames / float(framerate)
            # Allow small tolerances for floating point/date precision
            if not (0.99 <= duration <= 1.01):
                not_1s_files.append((fname, duration))
    except wave.Error as e:
        print(f"Error opening wave file '{fname}': {e}")
        not_1s_files.append((fname, 'unreadable'))

print(f"Total clip_filenames checked: {len(unique_filenames)}")

if missing_files:
    print(f"Missing wav files ({len(missing_files)}):")
    print(missing_files)
else:
    print("No missing wav files!")

if not_1s_files:
    print(f"Wavs not ~1 second long ({len(not_1s_files)}):")
    for fname, duration in not_1s_files:
        print(f"{fname}: duration={duration}")
else:
    print("All wav files are approximately 1 second long!")



No duplicate clip_filenames in merged_df.
Total clip_filenames checked: 11609
No missing wav files!
Wavs not ~1 second long (1):
BSM_20220731_17130300.wav: duration=0.864625


In [147]:
# Remove all rows in merged_df whose clip_filename is in not_1s_files list
# not_1s_files is a list of (fname, duration) pairs; we want only the filenames
not_1s_fnames = [fname for fname, _ in not_1s_files]
initial_row_count = len(merged_df)
merged_df = merged_df[~merged_df['clip_filename'].isin(not_1s_fnames)].reset_index(drop=True)
print(f"Removed {initial_row_count - len(merged_df)} rows with wav files not ~1s long.")


Removed 1 rows with wav files not ~1s long.


In [151]:
merged_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s
0,BSM_20170724_09480100.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 09:48:00.000,,,BSM,,201359382,-172.7,2017-07-24 09:48:01.000,2017-07-24 09:48:02.000,BSM_20170724_094800.wav,0.0,1.0
1,BSM_20170724_09480200.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 09:48:00.000,,,BSM,,201359382,-172.7,2017-07-24 09:48:02.000,2017-07-24 09:48:03.000,BSM_20170724_094800.wav,1.0,2.0
2,BSM_20170724_09480300.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 09:48:00.000,,,BSM,,201359382,-172.7,2017-07-24 09:48:03.000,2017-07-24 09:48:04.000,BSM_20170724_094800.wav,2.0,3.0
3,BSM_20170724_10580100.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 10:58:00.000,,,BSM,,201359382,-172.7,2017-07-24 10:58:01.000,2017-07-24 10:58:02.000,BSM_20170724_105800.wav,0.0,1.0
4,BSM_20170724_10580200.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,,,2017-07-24 10:58:00.000,,,BSM,,201359382,-172.7,2017-07-24 10:58:02.000,2017-07-24 10:58:03.000,BSM_20170724_105800.wav,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11603,RDL_20200906_14555938.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:55:57.380000,,,RDL,,5675,-176.5,2020-09-06 14:55:59.380000,2020-09-06 14:56:00.380000,RDL_20200906_14555738.wav,1.0,2.0
11604,RDL_20200906_14560038.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:55:57.380000,,,RDL,,5675,-176.5,2020-09-06 14:56:00.380000,2020-09-06 14:56:01.380000,RDL_20200906_14555738.wav,2.0,3.0
11605,RDL_20200906_14573769.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:57:36.690000,,,RDL,,5675,-176.5,2020-09-06 14:57:37.690000,2020-09-06 14:57:38.690000,RDL_20200906_14573669.wav,0.0,1.0
11606,RDL_20200906_14573869.wav,0,0,0,0,0,,old_abs_from_3s_snippets,,,,,,5675.200906144958.wav,,2020-09-06 14:57:36.690000,,,RDL,,5675,-176.5,2020-09-06 14:57:38.690000,2020-09-06 14:57:39.690000,RDL_20200906_14573669.wav,1.0,2.0


In [152]:
merged_df.groupby("labeling_effort")["BBPC"].value_counts()

labeling_effort           BBPC
evaluation_v1             0       3393
                          1        208
irene_20min               0        661
                          1        535
old_abs_CAC_last_samples  0        159
old_abs_from_3s_snippets  0       6652
Name: count, dtype: int64

In [ ]:
# merged_df.to_csv("../../data/Verified_Dataset/labels/labels_merged.csv", index=False)